# Анализ Training

Этот ноутбук предназначен для мониторинга хода обучения (training) и анализа радиальных функций распределения (RDF).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import glob
import re

TRAINING_DIR = ""
CONFIGS_DIR = "configs/methanol-data"
SYSTEMS = ["10CH3OH", "40CH3OH", "60CH3OH", "100CH3OH"]

SKIP_FIRST = 0


def get_latest_summary():
    files = glob.glob(os.path.join(TRAINING_DIR, "training_*_summary.csv"))
    if not files:
        return None
    return max(files, key=os.path.getmtime)


def get_available_iterations():
    files = glob.glob(os.path.join(TRAINING_DIR, "model-iter-*.bson"))
    iters = []
    for f in files:
        match = re.search(r"model-iter-(\d+)\.bson", f)
        if match:
            iters.append(int(match.group(1)))
    return sorted(iters)


def plot_metrics(df, skip_first=0):
    df = df.iloc[skip_first:]
    fig, axes = plt.subplots(3, 1, figsize=(12, 15), sharex=True)

    axes[0].plot(df["epoch"], df["mae"], marker="o", color="blue", label="MAE")
    axes[0].set_ylabel("MAE")
    axes[0].set_title("Training Metrics")
    axes[0].grid(True)

    axes[1].plot(
        df["epoch"], df["grad_norm"], marker="o", color="red", label="Grad Norm"
    )
    axes[1].set_ylabel("Gradient Norm")
    axes[1].set_yscale("log")
    axes[1].grid(True)

    axes[2].plot(
        df["epoch"], df["lr"], marker="o", color="green", label="Learning Rate"
    )
    axes[2].set_ylabel("Learning Rate")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()


def load_rdf(system, iteration):
    filename = os.path.join(TRAINING_DIR, f"RDFNN-{system}-CG-iter-{iteration:02d}.dat")
    if os.path.exists(filename):
        return np.loadtxt(filename)
    return None


def load_reference_rdf(system):
    filename = os.path.join(CONFIGS_DIR, system, f"{system}-CG.rdf")
    if os.path.exists(filename):
        return np.loadtxt(filename)
    return None

## Метрики обучения

In [ ]:
summary_file = get_latest_summary()
if summary_file:
    print(f"Loading summary from: {summary_file}")
    df = pd.read_csv(
        summary_file,
        comment="#",
        header=None,
        names=["epoch", "mae", "grad_norm", "lr"],
    )
    display(df.tail())
    plot_metrics(df, skip_first=SKIP_FIRST)
else:
    print("No summary file found in run_pmf_training")

## Анализ RDF

Сравнение рассчитанных RDF с референсными данными для всех доступных эпох.

In [ ]:
available_iters = get_available_iterations()
print(f"Available iterations: {available_iters}")

for system in SYSTEMS:
    ref_rdf = load_reference_rdf(system)
    if ref_rdf is None:
        print(f"Reference RDF not found for {system}")
        continue

    plt.figure(figsize=(12, 8))
    plt.plot(
        ref_rdf[:, 0], ref_rdf[:, 1], "k--", label="Reference", linewidth=2, alpha=0.7
    )

    colors = plt.cm.viridis(np.linspace(0, 1, len(available_iters[SKIP_FIRST:])))

    for i, iteration in enumerate(available_iters[SKIP_FIRST:]):
        sim_rdf = load_rdf(system, iteration)
        if sim_rdf is not None:
            plt.plot(
                sim_rdf[:, 0],
                sim_rdf[:, 1],
                label=f"Iter {iteration}",
                color=colors[i],
                alpha=0.8,
            )

    plt.title(f"RDF Comparison: {system}")
    plt.xlabel("r, Å")
    plt.ylabel("g(r)")
    plt.xlim(0.0, 15.0)
    plt.ylim(0.0, 1.8)
    plt.legend(ncol=2)
    plt.grid(True, alpha=0.3)
    plt.show()